In [0]:
# ============================================================
# BLOCO 1 — CONFIGURAÇÃO, LEITURA E INSPEÇÃO DA BRONZE
# ============================================================

CATALOG = "coretec_dev"
BRONZE_SCHEMA = "bronze"
SOURCE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.acessos_portaria"

# Define o contexto atual do Unity Catalog
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")

# Leitura da tabela Delta registrada no Unity Catalog
df_acessos_bronze = spark.table(SOURCE_TABLE)

print(f"Tabela de origem: {SOURCE_TABLE}")
print(f"Catálogo atual: {spark.catalog.currentCatalog()}")
print(f"Schema atual: {spark.catalog.currentDatabase()}")

# Inspeção do schema
print("\nSchema da tabela:")
df_acessos_bronze.printSchema()

# count() é uma ação Spark e executa a leitura dos dados
quantidade_bronze = df_acessos_bronze.count()

print(f"\nQuantidade de registros: {quantidade_bronze}")
print(f"Quantidade de colunas: {len(df_acessos_bronze.columns)}")
print(f"Colunas: {df_acessos_bronze.columns}")

# display() também é uma ação Spark
display(df_acessos_bronze.limit(10))

In [0]:
# ============================================================
# VALIDAÇÃO 1 — COMPLETUDE
# Nulos, strings vazias e espaços em branco
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

string_columns = [
    field.name
    for field in df_acessos_bronze.schema.fields
    if isinstance(field.dataType, StringType)
]

quality_expressions = []

for column in df_acessos_bronze.columns:

    # Quantidade de valores nulos
    quality_expressions.append(
        F.sum(
            F.when(F.col(column).isNull(), 1).otherwise(0)
        ).alias(f"{column}_nulos")
    )

    # Strings vazias ou contendo somente espaços
    if column in string_columns:
        quality_expressions.append(
            F.sum(
                F.when(
                    F.col(column).isNotNull()
                    & (F.trim(F.col(column)) == ""),
                    1
                ).otherwise(0)
            ).alias(f"{column}_vazios")
        )

resultado_completude = (
    df_acessos_bronze
    .agg(*quality_expressions)
    .first()
)

perfil_completude = []

for field in df_acessos_bronze.schema.fields:

    coluna = field.name

    quantidade_nulos = int(
        resultado_completude[f"{coluna}_nulos"] or 0
    )

    quantidade_vazios = (
        int(resultado_completude[f"{coluna}_vazios"] or 0)
        if coluna in string_columns
        else 0
    )

    perfil_completude.append(
        (
            coluna,
            field.dataType.simpleString(),
            quantidade_nulos,
            quantidade_vazios,
            quantidade_nulos + quantidade_vazios
        )
    )

df_completude = spark.createDataFrame(
    perfil_completude,
    [
        "coluna",
        "tipo_atual",
        "quantidade_nulos",
        "quantidade_strings_vazias",
        "total_ausencias"
    ]
)

display(
    df_completude.orderBy(
        F.desc("total_ausencias"),
        F.asc("coluna")
    )
)

In [0]:
# ============================================================
# VALIDAÇÃO 2 — UNICIDADE DA CHAVE CANDIDATA id_acesso
# ============================================================

from pyspark.sql import functions as F

resumo_id_acesso = (
    df_acessos_bronze
    .agg(
        F.count("*").alias("total_registros"),

        F.count("id_acesso").alias(
            "id_acesso_nao_nulos"
        ),

        F.countDistinct("id_acesso").alias(
            "id_acesso_distintos"
        ),

        F.sum(
            F.when(
                F.col("id_acesso").isNull()
                | (F.trim(F.col("id_acesso")) == ""),
                1
            ).otherwise(0)
        ).alias("id_acesso_ausentes")
    )
)

df_ids_duplicados = (
    df_acessos_bronze
    .filter(
        F.col("id_acesso").isNotNull()
        & (F.trim(F.col("id_acesso")) != "")
    )
    .groupBy("id_acesso")
    .count()
    .filter(F.col("count") > 1)
)

display(resumo_id_acesso)



In [0]:
# ============================================================
# VALIDAÇÃO 3 — INSPEÇÃO DOS IDs DUPLICADOS
# ============================================================

colunas_negocio = [
    "id_acesso",
    "id_condominio",
    "id_operador",
    "data_hora_acesso",
    "tipo_pessoa",
    "tipo_acesso",
    "status_acesso",
    "canal_autorizacao",
    "tempo_atendimento_segundos"
]

colunas_tecnicas = [
    "_source_file",
    "_ingestion_timestamp",
    "_ingestion_date",
    "_batch_id"
]

df_registros_duplicados = (
    df_acessos_bronze
    .join(
        df_ids_duplicados.select("id_acesso"),
        on="id_acesso",
        how="inner"
    )
    .select(
        *colunas_negocio,
        *colunas_tecnicas
    )
    .orderBy(
        "id_acesso",
        "_ingestion_timestamp",
        "_source_file"
    )
)

display(df_registros_duplicados)

In [0]:
# ============================================================
# VALIDAÇÃO 4 — CLASSIFICAÇÃO DAS DUPLICIDADES
# ============================================================

from pyspark.sql import functions as F

colunas_negocio_sem_id = [
    "id_condominio",
    "id_operador",
    "data_hora_acesso",
    "tipo_pessoa",
    "tipo_acesso",
    "status_acesso",
    "canal_autorizacao",
    "tempo_atendimento_segundos"
]

# Cria uma representação conjunta dos valores de negócio
df_classificacao_duplicidades = (
    df_registros_duplicados
    .groupBy("id_acesso")
    .agg(
        F.count("*").alias("quantidade_registros"),

        F.countDistinct(
            F.struct(
                *[
                    F.col(coluna)
                    for coluna in colunas_negocio_sem_id
                ]
            )
        ).alias("versoes_distintas_negocio"),

        F.countDistinct("_source_file").alias(
            "quantidade_arquivos_origem"
        )
    )
    .withColumn(
        "classificacao",
        F.when(
            F.col("versoes_distintas_negocio") == 1,
            F.lit("DUPLICIDADE_TECNICA")
        ).otherwise(
            F.lit("CONFLITO_DE_NEGOCIO")
        )
    )
    .orderBy("id_acesso")
)

display(df_classificacao_duplicidades)

In [0]:
# ============================================================
# VALIDAÇÃO 5 — VALIDADE DE data_hora_acesso
# ============================================================

from pyspark.sql import functions as F

# Teste de conversão sem alterar o DataFrame Bronze
df_validacao_data = (
    df_acessos_bronze
    .withColumn(
        "data_hora_convertida_teste",
        F.expr(
            """
            try_to_timestamp(
                trim(data_hora_acesso),
                'yyyy-MM-dd HH:mm:ss'
            )
            """
        )
    )
    .withColumn(
        "status_validacao_data",
        F.when(
            F.col("data_hora_acesso").isNull(),
            F.lit("NULO")
        )
        .when(
            F.trim(F.col("data_hora_acesso")) == "",
            F.lit("VAZIO")
        )
        .when(
            F.col("data_hora_convertida_teste").isNull(),
            F.lit("FORMATO_INVALIDO")
        )
        .otherwise(
            F.lit("VALIDO")
        )
    )
)

# Resumo da validação
display(
    df_validacao_data
    .groupBy("status_validacao_data")
    .count()
    .orderBy(F.desc("count"))
)

# Detalhamento apenas dos registros inválidos
print("Registros com data_hora_acesso inválida:")

display(
    df_validacao_data
    .filter(
        F.col("status_validacao_data") != "VALIDO"
    )
    .select(
        "id_acesso",
        "data_hora_acesso",
        "data_hora_convertida_teste",
        "status_validacao_data",
        "_source_file",
        "_ingestion_timestamp",
        "_batch_id"
    )
    .orderBy("id_acesso")
)

In [0]:
# ============================================================
# VALIDAÇÃO 6 — VALIDADE DE tempo_atendimento_segundos
# ============================================================

from pyspark.sql import functions as F

resumo_tempo_atendimento = (
    df_acessos_bronze
    .agg(
        F.count("*").alias("total_registros"),

        F.sum(
            F.when(
                F.col("tempo_atendimento_segundos").isNull(),
                1
            ).otherwise(0)
        ).alias("tempos_nulos"),

        F.sum(
            F.when(
                F.col("tempo_atendimento_segundos") < 0,
                1
            ).otherwise(0)
        ).alias("tempos_negativos"),

        F.sum(
            F.when(
                F.col("tempo_atendimento_segundos") == 0,
                1
            ).otherwise(0)
        ).alias("tempos_iguais_zero"),

        F.min("tempo_atendimento_segundos").alias(
            "tempo_minimo"
        ),

        F.max("tempo_atendimento_segundos").alias(
            "tempo_maximo"
        ),

        F.round(
            F.avg("tempo_atendimento_segundos"),
            2
        ).alias("tempo_medio")
    )
)

display(resumo_tempo_atendimento)


print("Registros com tempo de atendimento inválido:")

display(
    df_acessos_bronze
    .filter(
        F.col("tempo_atendimento_segundos").isNull()
        | (F.col("tempo_atendimento_segundos") < 0)
    )
    .select(
        "id_acesso",
        "data_hora_acesso",
        "tempo_atendimento_segundos",
        "_source_file",
        "_ingestion_timestamp",
        "_batch_id"
    )
    .orderBy("id_acesso")
)

In [0]:
# ============================================================
# VALIDAÇÃO 7 — REGISTRO SEM id_acesso
# ============================================================

from pyspark.sql import functions as F

df_sem_id_acesso = (
    df_acessos_bronze
    .filter(
        F.col("id_acesso").isNull()
        | (F.trim(F.col("id_acesso")) == "")
    )
)

print(
    "Quantidade de registros sem id_acesso:",
    df_sem_id_acesso.count()
)

display(
    df_sem_id_acesso.select(
        "id_acesso",
        "id_condominio",
        "id_operador",
        "data_hora_acesso",
        "tipo_pessoa",
        "tipo_acesso",
        "status_acesso",
        "canal_autorizacao",
        "tempo_atendimento_segundos",
        "_source_file",
        "_ingestion_timestamp",
        "_ingestion_date",
        "_batch_id"
    )
)

In [0]:
# ============================================================
# VALIDAÇÃO 8 — POSSÍVEL DUPLICIDADE DO REGISTRO SEM ID
# ============================================================

from pyspark.sql import functions as F

colunas_comparacao_sem_id = [
    "id_condominio",
    "id_operador",
    "data_hora_acesso",
    "tipo_pessoa",
    "tipo_acesso",
    "status_acesso",
    "canal_autorizacao",
    "tempo_atendimento_segundos"
]

# Atributos do registro que não possui id_acesso
df_atributos_sem_id = (
    df_sem_id_acesso
    .select(*colunas_comparacao_sem_id)
    .distinct()
)

# Procura registros com exatamente os mesmos atributos de negócio
df_possiveis_correspondencias = (
    df_acessos_bronze.alias("bronze")
    .join(
        df_atributos_sem_id.alias("sem_id"),
        on=colunas_comparacao_sem_id,
        how="inner"
    )
    .select(
        "bronze.id_acesso",
        *[
            F.col(f"bronze.{coluna}").alias(coluna)
            for coluna in colunas_comparacao_sem_id
        ],
        F.col("bronze._source_file").alias("_source_file"),
        F.col("bronze._ingestion_timestamp").alias(
            "_ingestion_timestamp"
        ),
        F.col("bronze._batch_id").alias("_batch_id")
    )
    .orderBy("id_acesso")
)

print(
    "Quantidade de registros com os mesmos atributos:",
    df_possiveis_correspondencias.count()
)

display(df_possiveis_correspondencias)

In [0]:
# ============================================================
# VALIDAÇÃO 9 — CONSISTÊNCIA DAS COLUNAS CATEGÓRICAS
# ============================================================

from pyspark.sql import functions as F

colunas_categoricas = [
    "tipo_pessoa",
    "tipo_acesso",
    "status_acesso",
    "canal_autorizacao"
]

df_distribuicao_categorias = (
    df_acessos_bronze
    .selectExpr(
        """
        stack(
            4,
            'tipo_pessoa', tipo_pessoa,
            'tipo_acesso', tipo_acesso,
            'status_acesso', status_acesso,
            'canal_autorizacao', canal_autorizacao
        ) AS (coluna, valor_original)
        """
    )
    .withColumn(
        "valor_sem_espacos",
        F.trim(F.col("valor_original"))
    )
    .withColumn(
        "valor_normalizado_teste",
        F.lower(F.trim(F.col("valor_original")))
    )
    .groupBy(
        "coluna",
        "valor_original",
        "valor_sem_espacos",
        "valor_normalizado_teste"
    )
    .count()
    .orderBy(
        "coluna",
        F.desc("count"),
        "valor_original"
    )
)

display(df_distribuicao_categorias)

In [0]:
# ============================================================
# VALIDAÇÃO 10 — CONFORMIDADE DOS IDENTIFICADORES
# ============================================================

from pyspark.sql import functions as F

colunas_identificadores = [
    "id_acesso",
    "id_condominio",
    "id_operador"
]

df_padrao_identificadores = (
    df_acessos_bronze
    .selectExpr(
        """
        stack(
            3,
            'id_acesso', id_acesso,
            'id_condominio', id_condominio,
            'id_operador', id_operador
        ) AS (coluna, identificador)
        """
    )
    .withColumn(
        "padrao_identificador",
        F.when(
            F.col("identificador").isNull(),
            F.lit("<NULL>")
        )
        .when(
            F.trim(F.col("identificador")) == "",
            F.lit("<VAZIO>")
        )
        .otherwise(
            F.regexp_replace(
                F.trim(F.col("identificador")),
                r"\d",
                "9"
            )
        )
    )
    .groupBy(
        "coluna",
        "padrao_identificador"
    )
    .count()
    .orderBy(
        "coluna",
        F.desc("count"),
        "padrao_identificador"
    )
)

display(df_padrao_identificadores)

In [0]:
# ============================================================
# VALIDAÇÃO 11 — REGRAS DE CONFORMIDADE DOS IDENTIFICADORES
# ============================================================

from pyspark.sql import functions as F

resumo_conformidade_ids = (
    df_acessos_bronze
    .agg(
        F.sum(
            F.when(
                F.col("id_acesso").isNotNull()
                & ~F.trim(F.col("id_acesso")).rlike(r"^ACS\d{8}$"),
                1
            ).otherwise(0)
        ).alias("id_acesso_formato_invalido"),

        F.sum(
            F.when(
                F.col("id_condominio").isNull()
                | ~F.trim(F.col("id_condominio")).rlike(r"^COND\d{3}$"),
                1
            ).otherwise(0)
        ).alias("id_condominio_formato_invalido"),

        F.sum(
            F.when(
                F.col("id_operador").isNull()
                | ~F.trim(F.col("id_operador")).rlike(r"^OPR\d{3}$"),
                1
            ).otherwise(0)
        ).alias("id_operador_formato_invalido")
    )
)

display(resumo_conformidade_ids)

In [0]:
# ============================================================
# VALIDAÇÃO 12 — LINEAGE E CONSISTÊNCIA POR ARQUIVO DE ORIGEM
# ============================================================

from pyspark.sql import functions as F

df_lineage_arquivo = (
    df_validacao_data
    .groupBy(
        "_source_file",
        "_batch_id"
    )
    .agg(
        F.count("*").alias("quantidade_registros"),

        F.min("data_hora_convertida_teste").alias(
            "menor_data_hora"
        ),

        F.max("data_hora_convertida_teste").alias(
            "maior_data_hora"
        ),

        F.sum(
            F.when(
                F.col("status_validacao_data") != "VALIDO",
                1
            ).otherwise(0)
        ).alias("datas_invalidas"),

        F.countDistinct("id_acesso").alias(
            "ids_acesso_distintos"
        )
    )
    .orderBy("_source_file")
)

display(df_lineage_arquivo)

In [0]:
# ============================================================
# VALIDAÇÃO 13 — MÊS DO EVENTO VERSUS MÊS DO ARQUIVO
# ============================================================

from pyspark.sql import functions as F

df_validacao_mes_arquivo = (
    df_validacao_data
    # Extrai 2026-01, 2026-02 etc. do nome do arquivo
    .withColumn(
        "mes_referencia_arquivo",
        F.regexp_extract(
            F.col("_source_file"),
            r"acessos_portaria_(\d{4}-\d{2})\.parquet",
            1
        )
    )
    # Obtém o mês do evento após a conversão válida para timestamp
    .withColumn(
        "mes_evento",
        F.date_format(
            F.col("data_hora_convertida_teste"),
            "yyyy-MM"
        )
    )
    .withColumn(
        "status_mes_arquivo",
        F.when(
            F.col("data_hora_convertida_teste").isNull(),
            F.lit("DATA_INVALIDA")
        )
        .when(
            F.col("mes_evento") == F.col("mes_referencia_arquivo"),
            F.lit("MES_COMPATIVEL")
        )
        .otherwise(
            F.lit("MES_DIVERGENTE")
        )
    )
)

print("Resumo da consistência entre evento e arquivo:")

display(
    df_validacao_mes_arquivo
    .groupBy(
        "mes_referencia_arquivo",
        "status_mes_arquivo"
    )
    .count()
    .orderBy(
        "mes_referencia_arquivo",
        "status_mes_arquivo"
    )
)

print("Registros com mês divergente ou data inválida:")

display(
    df_validacao_mes_arquivo
    .filter(
        F.col("status_mes_arquivo") != "MES_COMPATIVEL"
    )
    .select(
        "id_acesso",
        "data_hora_acesso",
        "mes_evento",
        "mes_referencia_arquivo",
        "status_mes_arquivo",
        "_source_file",
        "_batch_id"
    )
    .orderBy(
        "mes_referencia_arquivo",
        "id_acesso"
    )
)

In [0]:
# ============================================================
# TRANSFORMAÇÃO 1 — PREPARAÇÃO E FLAGS DE QUALIDADE
# ============================================================

from pyspark.sql import functions as F


df_acessos_preparado = (
    df_acessos_bronze

    # Seleção e tratamento básico das colunas
    .select(
        F.trim(F.col("id_acesso")).alias("id_acesso"),
        F.trim(F.col("id_condominio")).alias("id_condominio"),
        F.trim(F.col("id_operador")).alias("id_operador"),

        # Preservamos temporariamente o valor original
        F.trim(F.col("data_hora_acesso")).alias(
            "_raw_data_hora_acesso"
        ),

        F.trim(F.col("tipo_pessoa")).alias("tipo_pessoa"),
        F.trim(F.col("tipo_acesso")).alias("tipo_acesso"),
        F.trim(F.col("status_acesso")).alias("status_acesso"),
        F.trim(F.col("canal_autorizacao")).alias(
            "canal_autorizacao"
        ),

        F.col("tempo_atendimento_segundos")
        .cast("int")
        .alias("tempo_atendimento_segundos"),

        # Metadados provenientes da Bronze
        F.col("_source_file"),
        F.col("_ingestion_timestamp"),
        F.col("_ingestion_date"),
        F.col("_batch_id")
    )

    # Conversão segura da data para timestamp
    .withColumn(
        "data_hora_acesso",
        F.expr(
            """
            try_to_timestamp(
                _raw_data_hora_acesso,
                'yyyy-MM-dd HH:mm:ss'
            )
            """
        )
    )

    # Mês declarado no nome do arquivo
    .withColumn(
        "_mes_referencia_arquivo",
        F.regexp_extract(
            F.col("_source_file"),
            r"acessos_portaria_(\d{4}-\d{2})\.parquet",
            1
        )
    )

    # Mês efetivo do evento
    .withColumn(
        "_mes_evento",
        F.date_format(
            F.col("data_hora_acesso"),
            "yyyy-MM"
        )
    )

    # Flags de qualidade
    .withColumn(
        "_dq_id_acesso_preenchido",
        F.col("id_acesso").isNotNull()
        & (F.col("id_acesso") != "")
    )
    .withColumn(
        "_dq_data_hora_valida",
        F.col("data_hora_acesso").isNotNull()
    )
    .withColumn(
        "_dq_tempo_atendimento_valido",
        F.col("tempo_atendimento_segundos").isNotNull()
        & (F.col("tempo_atendimento_segundos") >= 0)
    )
    .withColumn(
        "_dq_mes_compativel_arquivo",
        F.col("_mes_evento")
        == F.col("_mes_referencia_arquivo")
    )

    # Pontuação usada posteriormente para escolher a melhor versão
    .withColumn(
        "_dq_score",
        F.col("_dq_id_acesso_preenchido").cast("int")
        + F.col("_dq_data_hora_valida").cast("int")
        + F.col("_dq_tempo_atendimento_valido").cast("int")
        + F.col("_dq_mes_compativel_arquivo").cast("int")
    )
)


print("Resumo das combinações de qualidade:")

display(
    df_acessos_preparado
    .groupBy(
        "_dq_id_acesso_preenchido",
        "_dq_data_hora_valida",
        "_dq_tempo_atendimento_valido",
        "_dq_mes_compativel_arquivo",
        "_dq_score"
    )
    .count()
    .orderBy(
        F.asc("_dq_score"),
        F.desc("count")
    )
)


print("Registros com alguma falha de qualidade:")

display(
    df_acessos_preparado
    .filter(F.col("_dq_score") < 4)
    .select(
        "id_acesso",
        "_raw_data_hora_acesso",
        "data_hora_acesso",
        "tempo_atendimento_segundos",
        "_dq_id_acesso_preenchido",
        "_dq_data_hora_valida",
        "_dq_tempo_atendimento_valido",
        "_dq_mes_compativel_arquivo",
        "_dq_score",
        "_source_file"
    )
    .orderBy(
        F.asc("_dq_score"),
        F.asc("id_acesso")
    )
)

In [0]:
# ============================================================
# DIAGNÓSTICO — VERIFICAR ACS00000053 NO DATAFRAME PREPARADO
# ============================================================

from pyspark.sql import functions as F

display(
    df_acessos_preparado
    .filter(F.col("id_acesso") == "ACS00000053")
    .select(
        "id_acesso",
        "_raw_data_hora_acesso",
        "data_hora_acesso",
        "tempo_atendimento_segundos",
        "_mes_evento",
        "_mes_referencia_arquivo",
        "_dq_id_acesso_preenchido",
        "_dq_data_hora_valida",
        "_dq_tempo_atendimento_valido",
        "_dq_mes_compativel_arquivo",
        "_dq_score",
        "_source_file"
    )
    .orderBy("_raw_data_hora_acesso")
)

In [0]:
# ============================================================
# CORREÇÃO — FLAG DE MÊS E SCORE DE QUALIDADE
# ============================================================

from pyspark.sql import functions as F

df_acessos_preparado = (
    df_acessos_preparado

    .withColumn(
        "_dq_mes_compativel_arquivo",
        F.when(
            F.col("_mes_evento").isNull(),
            F.lit(False)
        )
        .otherwise(
            F.col("_mes_evento")
            == F.col("_mes_referencia_arquivo")
        )
    )

    .withColumn(
        "_dq_score",
        F.col("_dq_id_acesso_preenchido").cast("int")
        + F.col("_dq_data_hora_valida").cast("int")
        + F.col("_dq_tempo_atendimento_valido").cast("int")
        + F.col("_dq_mes_compativel_arquivo").cast("int")
    )
)

display(
    df_acessos_preparado
    .filter(F.col("id_acesso") == "ACS00000053")
    .select(
        "id_acesso",
        "_raw_data_hora_acesso",
        "data_hora_acesso",
        "_mes_evento",
        "_mes_referencia_arquivo",
        "_dq_data_hora_valida",
        "_dq_mes_compativel_arquivo",
        "_dq_score"
    )
    .orderBy("_raw_data_hora_acesso")
)

In [0]:
# ============================================================
# TRANSFORMAÇÃO 2 — RANKING E SELEÇÃO DA MELHOR VERSÃO
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window


# Registros com chave preenchida podem participar da deduplicação
df_acessos_com_chave = (
    df_acessos_preparado
    .filter(
        F.col("id_acesso").isNotNull()
        & (F.col("id_acesso") != "")
    )
)


# Registro sem chave fica separado para tratamento controlado
df_acessos_sem_chave = (
    df_acessos_preparado
    .filter(
        F.col("id_acesso").isNull()
        | (F.col("id_acesso") == "")
    )
)


janela_deduplicacao = (
    Window
    .partitionBy("id_acesso")
    .orderBy(
        F.desc("_dq_score"),
        F.desc("_ingestion_timestamp"),
        F.asc("_source_file")
    )
)


df_acessos_rankeado = (
    df_acessos_com_chave
    .withColumn(
        "_row_number",
        F.row_number().over(janela_deduplicacao)
    )
)


# Melhor versão de cada id_acesso
df_acessos_selecionado = (
    df_acessos_rankeado
    .filter(F.col("_row_number") == 1)
)


# Versões não selecionadas
df_acessos_nao_selecionado = (
    df_acessos_rankeado
    .filter(F.col("_row_number") > 1)
)


print("Resumo da seleção:")

display(
    spark.createDataFrame(
        [
            (
                df_acessos_preparado.count(),
                df_acessos_com_chave.count(),
                df_acessos_sem_chave.count(),
                df_acessos_selecionado.count(),
                df_acessos_nao_selecionado.count()
            )
        ],
        [
            "registros_preparados",
            "registros_com_chave",
            "registros_sem_chave",
            "registros_selecionados",
            "versoes_nao_selecionadas"
        ]
    )
)


print("Decisão tomada para os IDs duplicados:")

display(
    df_acessos_rankeado
    .filter(
        F.col("id_acesso").isin(
            "ACS00000001",
            "ACS00000002",
            "ACS00000004",
            "ACS00000053",
            "ACS00000061"
        )
    )
    .select(
        "id_acesso",
        "_raw_data_hora_acesso",
        "data_hora_acesso",
        "tempo_atendimento_segundos",
        "_dq_data_hora_valida",
        "_dq_tempo_atendimento_valido",
        "_dq_mes_compativel_arquivo",
        "_dq_score",
        "_source_file",
        "_row_number"
    )
    .orderBy(
        "id_acesso",
        "_row_number"
    )
)

In [0]:
# ============================================================
# VALIDAÇÃO 14 — RESULTADO DA DEDUPLICAÇÃO E REJEIÇÕES
# ============================================================

from pyspark.sql import functions as F


# Registros duplicados/conflitantes que não foram selecionados
df_acessos_rejeitados_duplicidade = (
    df_acessos_nao_selecionado
    .withColumn(
        "_rejection_reason",
        F.when(
            ~F.col("_dq_data_hora_valida"),
            F.lit("DATA_HORA_INVALIDA")
        )
        .when(
            ~F.col("_dq_tempo_atendimento_valido"),
            F.lit("TEMPO_ATENDIMENTO_INVALIDO")
        )
        .when(
            ~F.col("_dq_mes_compativel_arquivo"),
            F.lit("DUPLICIDADE_TECNICA_ENTRE_ARQUIVOS")
        )
        .otherwise(
            F.lit("VERSAO_DUPLICADA_NAO_SELECIONADA")
        )
    )
)


# Registro sem chave, já confirmado como cópia de ACS00000008
df_acessos_rejeitados_sem_chave = (
    df_acessos_sem_chave
    .withColumn(
        "_rejection_reason",
        F.lit("REGISTRO_DUPLICADO_SEM_CHAVE")
    )
)


# União de todos os registros que não entrarão na Silver principal
df_acessos_rejeitados = (
    df_acessos_rejeitados_duplicidade
    .unionByName(
        df_acessos_rejeitados_sem_chave,
        allowMissingColumns=True
    )
)


# ------------------------------------------------------------
# 1. Validação da tabela principal selecionada
# ------------------------------------------------------------

resumo_selecao = (
    df_acessos_selecionado
    .agg(
        F.count("*").alias("registros_selecionados"),

        F.countDistinct("id_acesso").alias(
            "ids_acesso_distintos"
        ),

        F.sum(
            F.when(
                F.col("id_acesso").isNull()
                | (F.col("id_acesso") == ""),
                1
            ).otherwise(0)
        ).alias("ids_ausentes"),

        F.sum(
            F.when(
                ~F.col("_dq_data_hora_valida"),
                1
            ).otherwise(0)
        ).alias("datas_invalidas"),

        F.sum(
            F.when(
                ~F.col("_dq_tempo_atendimento_valido"),
                1
            ).otherwise(0)
        ).alias("tempos_invalidos"),

        F.min("_dq_score").alias("menor_score"),

        F.max("_dq_score").alias("maior_score")
    )
)

print("Validação dos registros selecionados:")

display(resumo_selecao)


# ------------------------------------------------------------
# 2. Distribuição dos registros rejeitados
# ------------------------------------------------------------

print("Motivos dos registros não selecionados:")

display(
    df_acessos_rejeitados
    .groupBy("_rejection_reason")
    .count()
    .orderBy("_rejection_reason")
)


print("Detalhamento dos registros não selecionados:")

display(
    df_acessos_rejeitados
    .select(
        "id_acesso",
        "_raw_data_hora_acesso",
        "tempo_atendimento_segundos",
        "_dq_score",
        "_rejection_reason",
        "_source_file",
        "_batch_id"
    )
    .orderBy(
        "_rejection_reason",
        "id_acesso"
    )
)

In [0]:
# ============================================================
# TRANSFORMAÇÃO 3 — DATAFRAMES FINAIS DA SILVER E REJEITADOS
# ============================================================

from pyspark.sql import functions as F


SILVER_BATCH_ID = (
    "acessos_portaria_silver_2026_06_21_batch_001"
)


# DataFrame principal da Silver
df_acessos_silver = (
    df_acessos_selecionado
    .select(
        # Colunas de negócio
        F.col("id_acesso"),
        F.col("id_condominio"),
        F.col("id_operador"),
        F.col("data_hora_acesso"),
        F.col("tipo_pessoa"),
        F.col("tipo_acesso"),
        F.col("status_acesso"),
        F.col("canal_autorizacao"),
        F.col("tempo_atendimento_segundos"),

        # Metadados preservados da Bronze
        F.col("_source_file"),
        F.col("_ingestion_timestamp"),
        F.col("_ingestion_date"),
        F.col("_batch_id")
    )
    .withColumn(
        "_processed_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_silver_batch_id",
        F.lit(SILVER_BATCH_ID)
    )
)


# DataFrame de rejeitados para auditoria
df_acessos_rejeitados_final = (
    df_acessos_rejeitados
    .select(
        F.col("id_acesso"),
        F.col("id_condominio"),
        F.col("id_operador"),

        # Preserva o valor original, inclusive DATA_INVALIDA
        F.col("_raw_data_hora_acesso").alias(
            "data_hora_acesso_original"
        ),

        F.col("tipo_pessoa"),
        F.col("tipo_acesso"),
        F.col("status_acesso"),
        F.col("canal_autorizacao"),
        F.col("tempo_atendimento_segundos"),

        F.col("_rejection_reason"),

        # Metadados de origem
        F.col("_source_file"),
        F.col("_ingestion_timestamp"),
        F.col("_ingestion_date"),
        F.col("_batch_id")
    )
    .withColumn(
        "_processed_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_silver_batch_id",
        F.lit(SILVER_BATCH_ID)
    )
)


print("Schema preparado para a Silver principal:")

df_acessos_silver.printSchema()

print(
    "Quantidade preparada para a Silver:",
    df_acessos_silver.count()
)


print("\nSchema preparado para os rejeitados:")

df_acessos_rejeitados_final.printSchema()

print(
    "Quantidade preparada como rejeitada:",
    df_acessos_rejeitados_final.count()
)


print("\nAmostra da Silver principal:")

display(
    df_acessos_silver
    .orderBy("id_acesso")
    .limit(10)
)

In [0]:
# ============================================================
# BLOCO 4 — VALIDAÇÃO FINAL ANTES DA GRAVAÇÃO
# ============================================================

from pyspark.sql import functions as F


# ------------------------------------------------------------
# 1. Reconciliação das quantidades
# ------------------------------------------------------------

quantidade_bronze = df_acessos_bronze.count()
quantidade_silver = df_acessos_silver.count()
quantidade_rejeitados = df_acessos_rejeitados_final.count()

df_reconciliacao = spark.createDataFrame(
    [
        (
            quantidade_bronze,
            quantidade_silver,
            quantidade_rejeitados,
            quantidade_silver + quantidade_rejeitados,
            quantidade_bronze
            == quantidade_silver + quantidade_rejeitados
        )
    ],
    [
        "quantidade_bronze",
        "quantidade_silver",
        "quantidade_rejeitados",
        "silver_mais_rejeitados",
        "reconciliacao_ok"
    ]
)

print("1. Reconciliação das quantidades:")
display(df_reconciliacao)


# ------------------------------------------------------------
# 2. Qualidade da Silver principal
# ------------------------------------------------------------

df_validacao_final = (
    df_acessos_silver
    .agg(
        F.count("*").alias("total_registros"),

        F.countDistinct("id_acesso").alias(
            "ids_acesso_distintos"
        ),

        F.sum(
            F.when(
                F.col("id_acesso").isNull()
                | (F.trim(F.col("id_acesso")) == ""),
                1
            ).otherwise(0)
        ).alias("ids_acesso_ausentes"),

        F.sum(
            F.when(
                F.col("data_hora_acesso").isNull(),
                1
            ).otherwise(0)
        ).alias("datas_invalidas_ou_nulas"),

        F.sum(
            F.when(
                F.col("tempo_atendimento_segundos").isNull()
                | (F.col("tempo_atendimento_segundos") < 0),
                1
            ).otherwise(0)
        ).alias("tempos_invalidos"),

        F.min("tempo_atendimento_segundos").alias(
            "tempo_minimo"
        ),

        F.max("tempo_atendimento_segundos").alias(
            "tempo_maximo"
        )
    )
    .withColumn(
        "duplicidades_id_acesso",
        F.col("total_registros")
        - F.col("ids_acesso_distintos")
    )
)

print("2. Validação de qualidade da Silver:")
display(df_validacao_final)


# ------------------------------------------------------------
# 3. Distribuição das categorias após a transformação
# ------------------------------------------------------------

df_categorias_silver = (
    df_acessos_silver
    .selectExpr(
        """
        stack(
            4,
            'tipo_pessoa', tipo_pessoa,
            'tipo_acesso', tipo_acesso,
            'status_acesso', status_acesso,
            'canal_autorizacao', canal_autorizacao
        ) AS (coluna, valor)
        """
    )
    .groupBy("coluna", "valor")
    .count()
    .orderBy(
        "coluna",
        F.desc("count"),
        "valor"
    )
)

print("3. Distribuição das categorias na Silver:")
display(df_categorias_silver)


# ------------------------------------------------------------
# 4. Schema final em memória
# ------------------------------------------------------------

print("4. Schema final preparado para gravação:")
df_acessos_silver.printSchema()

In [0]:
# ============================================================
# BLOCO 5 — GRAVAÇÃO DA TABELA DELTA SILVER
# ============================================================

from pyspark.sql import functions as F


SILVER_TABLE = "coretec_dev.silver.acessos_portaria"

# Captura um único timestamp para toda a execução Silver
processed_timestamp = (
    spark.sql("SELECT current_timestamp() AS timestamp")
    .first()["timestamp"]
)

# Fixa os metadados desta execução antes da gravação
df_acessos_silver_final = (
    df_acessos_silver
    .withColumn(
        "_processed_timestamp",
        F.lit(processed_timestamp).cast("timestamp")
    )
    .withColumn(
        "_silver_batch_id",
        F.lit(SILVER_BATCH_ID)
    )
)

# Grava como tabela Delta gerenciada pelo Unity Catalog
(
    df_acessos_silver_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"Tabela gravada com sucesso: {SILVER_TABLE}")
print(f"Silver batch ID: {SILVER_BATCH_ID}")
print(f"Processed timestamp: {processed_timestamp}")

In [0]:
# ============================================================
# VALIDAÇÃO PÓS-GRAVAÇÃO 1 — TABELA SILVER PERSISTIDA
# ============================================================

from pyspark.sql import functions as F

SILVER_TABLE = "coretec_dev.silver.acessos_portaria"

# Confirma que a tabela está registrada no Unity Catalog
tabela_existe = spark.catalog.tableExists(SILVER_TABLE)

print(f"Tabela existente: {tabela_existe}")

# Lê novamente a tabela persistida
df_acessos_silver_persistido = spark.table(SILVER_TABLE)

# Valida quantidade gravada
quantidade_persistida = df_acessos_silver_persistido.count()

print(f"Quantidade persistida: {quantidade_persistida}")

# Exibe o schema realmente salvo
print("\nSchema persistido:")
df_acessos_silver_persistido.printSchema()

# Valida os metadados específicos da execução Silver
print("\nMetadados da execução Silver:")

display(
    df_acessos_silver_persistido
    .agg(
        F.countDistinct("_silver_batch_id").alias(
            "quantidade_silver_batch_ids"
        ),
        F.first("_silver_batch_id").alias(
            "silver_batch_id"
        ),
        F.countDistinct("_processed_timestamp").alias(
            "quantidade_processed_timestamps"
        ),
        F.first("_processed_timestamp").alias(
            "processed_timestamp"
        )
    )
)

In [0]:
# ============================================================
# VALIDAÇÃO PÓS-GRAVAÇÃO 2 — DELTA, HISTÓRICO E AMOSTRA FINAL
# ============================================================

SILVER_TABLE = "coretec_dev.silver.acessos_portaria"


# ------------------------------------------------------------
# 1. Detalhes físicos e formato da tabela
# ------------------------------------------------------------

print("1. Detalhes da tabela persistida:")

df_detalhes_delta = spark.sql(
    f"DESCRIBE DETAIL {SILVER_TABLE}"
)

display(
    df_detalhes_delta.select(
        "format",
        "location",
        "numFiles",
        "sizeInBytes",
        "createdAt",
        "lastModified"
    )
)


# ------------------------------------------------------------
# 2. Histórico transacional Delta
# ------------------------------------------------------------

print("2. Histórico Delta:")

df_historico_delta = spark.sql(
    f"DESCRIBE HISTORY {SILVER_TABLE}"
)

display(
    df_historico_delta.select(
        "version",
        "timestamp",
        "operation",
        "operationParameters",
        "operationMetrics"
    )
    .orderBy("version", ascending=False)
)


# ------------------------------------------------------------
# 3. Versão atual
# ------------------------------------------------------------

versao_atual = (
    df_historico_delta
    .selectExpr("max(version) AS versao_atual")
    .first()["versao_atual"]
)

print(f"Versão atual da tabela Delta: {versao_atual}")


# ------------------------------------------------------------
# 4. Amostra final diretamente da tabela persistida
# ------------------------------------------------------------

print("3. Amostra final da Silver persistida:")

display(
    spark.table(SILVER_TABLE)
    .orderBy("id_acesso")
    .limit(10)
)

In [0]:
# ============================================================
# GRAVAÇÃO DOS REGISTROS REJEITADOS PARA AUDITORIA
# ============================================================

from pyspark.sql import functions as F

REJECTED_TABLE = "coretec_dev.ops.rejected_acessos_portaria"

# Utiliza o mesmo timestamp e batch da execução Silver principal
df_acessos_rejeitados_persistir = (
    df_acessos_rejeitados_final
    .withColumn(
        "_processed_timestamp",
        F.lit(processed_timestamp).cast("timestamp")
    )
    .withColumn(
        "_silver_batch_id",
        F.lit(SILVER_BATCH_ID)
    )
)

(
    df_acessos_rejeitados_persistir
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(REJECTED_TABLE)
)

print(f"Tabela de rejeitados gravada: {REJECTED_TABLE}")
print(
    "Quantidade de rejeitados gravados:",
    spark.table(REJECTED_TABLE).count()
)

In [0]:
# ============================================================
# VALIDAÇÃO FINAL — TABELA DE REJEITADOS
# ============================================================

from pyspark.sql import functions as F

REJECTED_TABLE = "coretec_dev.ops.rejected_acessos_portaria"

tabela_rejeitados_existe = spark.catalog.tableExists(REJECTED_TABLE)

df_rejeitados_persistido = spark.table(REJECTED_TABLE)

print(f"Tabela existente: {tabela_rejeitados_existe}")
print(f"Quantidade persistida: {df_rejeitados_persistido.count()}")

print("\nQuantidade por motivo de rejeição:")

display(
    df_rejeitados_persistido
    .groupBy("_rejection_reason")
    .count()
    .orderBy("_rejection_reason")
)

print("\nMetadados da execução:")

display(
    df_rejeitados_persistido
    .agg(
        F.countDistinct("_silver_batch_id").alias(
            "quantidade_silver_batch_ids"
        ),
        F.first("_silver_batch_id").alias(
            "silver_batch_id"
        ),
        F.countDistinct("_processed_timestamp").alias(
            "quantidade_processed_timestamps"
        ),
        F.first("_processed_timestamp").alias(
            "processed_timestamp"
        )
    )
)

print("\nFormato da tabela:")

display(
    spark.sql(f"DESCRIBE DETAIL {REJECTED_TABLE}")
    .select(
        "format",
        "numFiles",
        "sizeInBytes"
    )
)

In [0]:
# Leitura da tabela Bronze

OCCURRENCES_TABLE = "coretec_dev.bronze.ocorrencias"

df_ocorrencias_bronze = spark.table(OCCURRENCES_TABLE)

print(f"Tabela: {OCCURRENCES_TABLE}")
print(f"Quantidade de registros: {df_ocorrencias_bronze.count()}")
print(f"Quantidade de colunas: {len(df_ocorrencias_bronze.columns)}")

df_ocorrencias_bronze.printSchema()

display(df_ocorrencias_bronze.limit(10))

In [0]:
from pyspark.sql import functions as F

display(
    df_ocorrencias_bronze
    .orderBy("id_ocorrencia")
    .limit(10)
)

display(
    df_ocorrencias_bronze.agg(
        F.count("*").alias("total"),
        F.count("id_ocorrencia").alias("ids_nao_nulos"),
        F.countDistinct("id_ocorrencia").alias("ids_distintos")
    )
)

In [0]:
from pyspark.sql import functions as F

df_ids_ocorrencia_duplicados = (
    df_ocorrencias_bronze
    .filter(F.col("id_ocorrencia").isNotNull())
    .groupBy("id_ocorrencia")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"), "id_ocorrencia")
)

display(df_ids_ocorrencia_duplicados)

In [0]:
df_ocorrencias_duplicadas = (
    df_ocorrencias_bronze
    .join(
        df_ids_ocorrencia_duplicados.select("id_ocorrencia"),
        on="id_ocorrencia",
        how="left_semi"
    )
    .orderBy("id_ocorrencia", "_source_file")
)

display(df_ocorrencias_duplicadas)

In [0]:
from pyspark.sql import functions as F

df_ocorrencias_late = (
    df_ocorrencias_bronze
    .filter(F.lower(F.col("_source_file")).contains("late"))
)

df_ids_regulares = (
    df_ocorrencias_bronze
    .filter(~F.lower(F.col("_source_file")).contains("late"))
    .select("id_ocorrencia")
    .distinct()
    .withColumn("existe_em_arquivo_regular", F.lit(True))
)

df_analise_late = (
    df_ocorrencias_late
    .join(
        df_ids_regulares,
        on="id_ocorrencia",
        how="left"
    )
    .withColumn(
        "classificacao_late",
        F.when(
            F.col("existe_em_arquivo_regular") == True,
            "ID_JA_EXISTENTE"
        ).otherwise("NOVO_REGISTRO")
    )
)

display(df_analise_late.orderBy("id_ocorrencia"))

In [0]:
display(
    df_ocorrencias_bronze
    .filter(F.col("id_ocorrencia").isNull())
)

In [0]:
colunas_comparacao = [
    "id_condominio",
    "id_operador",
    "data_ocorrencia",
    "tipo_ocorrencia",
    "prioridade",
    "status_ocorrencia",
    "tempo_resolucao_minutos",
    "descricao_resumida"
]

df_ocorrencia_sem_id = (
    df_ocorrencias_bronze
    .filter(F.col("id_ocorrencia").isNull())
)

df_correspondencias_sem_id = (
    df_ocorrencias_bronze
    .join(
        df_ocorrencia_sem_id.select(*colunas_comparacao),
        on=colunas_comparacao,
        how="inner"
    )
    .orderBy("id_ocorrencia")
)

display(df_correspondencias_sem_id)

In [0]:
df_ocorrencias_validacao = (
    df_ocorrencias_bronze
    .withColumn(
        "data_ocorrencia_teste",
        F.expr(
            "try_to_timestamp(trim(data_ocorrencia), 'yyyy-MM-dd HH:mm:ss')"
        )
    )
    .withColumn(
        "tempo_resolucao_teste",
        F.expr(
            "try_cast(trim(tempo_resolucao_minutos) AS INT)"
        )
    )
)

display(
    df_ocorrencias_validacao.agg(
        F.sum(
            F.when(
                F.col("data_ocorrencia").isNotNull()
                & F.col("data_ocorrencia_teste").isNull(),
                1
            ).otherwise(0)
        ).alias("datas_invalidas"),

        F.sum(
            F.when(
                F.col("tempo_resolucao_minutos").isNotNull()
                & F.col("tempo_resolucao_teste").isNull(),
                1
            ).otherwise(0)
        ).alias("tempos_nao_convertidos"),

        F.sum(
            F.when(
                F.col("tempo_resolucao_teste") < 0,
                1
            ).otherwise(0)
        ).alias("tempos_negativos")
    )
)

display(
    df_ocorrencias_validacao
    .filter(
        F.col("data_ocorrencia_teste").isNull()
        | F.col("tempo_resolucao_teste").isNull()
        | (F.col("tempo_resolucao_teste") < 0)
    )
)

In [0]:
display(
    df_ocorrencias_validacao
    .filter(
        F.col("tempo_resolucao_minutos").isNotNull()
        & F.col("tempo_resolucao_teste").isNull()
    )
    .groupBy("tempo_resolucao_minutos")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
df_validacao_tempo = (
    df_ocorrencias_bronze
    .withColumn(
        "tempo_double",
        F.expr("try_cast(trim(tempo_resolucao_minutos) AS DOUBLE)")
    )
)

display(
    df_validacao_tempo.agg(
        F.sum(
            F.when(
                F.col("tempo_resolucao_minutos").isNotNull()
                & F.col("tempo_double").isNull(),
                1
            ).otherwise(0)
        ).alias("valores_nao_numericos"),

        F.sum(
            F.when(
                F.col("tempo_double") < 0,
                1
            ).otherwise(0)
        ).alias("valores_negativos"),

        F.sum(
            F.when(
                F.col("tempo_double") != F.floor(F.col("tempo_double")),
                1
            ).otherwise(0)
        ).alias("valores_com_casas_decimais")
    )
)

In [0]:
colunas_negocio = [
    "id_ocorrencia",
    "id_condominio",
    "id_operador",
    "data_ocorrencia",
    "tipo_ocorrencia",
    "prioridade",
    "status_ocorrencia",
    "tempo_resolucao_minutos",
    "descricao_resumida"
]

display(
    df_ocorrencias_bronze.agg(
        *[
            F.sum(
                F.when(
                    F.col(coluna).isNull()
                    | (F.trim(F.col(coluna)) == ""),
                    1
                ).otherwise(0)
            ).alias(coluna)
            for coluna in colunas_negocio
        ]
    )
)

In [0]:
display(
    df_ocorrencias_bronze
    .withColumn(
        "tempo_ausente",
        F.col("tempo_resolucao_minutos").isNull()
        | (F.trim(F.col("tempo_resolucao_minutos")) == "")
    )
    .groupBy("status_ocorrencia", "tempo_ausente")
    .count()
    .orderBy("status_ocorrencia", "tempo_ausente")
)

In [0]:
display(
    df_ocorrencias_bronze
    .groupBy("prioridade")
    .count()
    .orderBy(F.desc("count"))
)

display(
    df_ocorrencias_bronze
    .groupBy("status_ocorrencia")
    .count()
    .orderBy(F.desc("count"))
)

display(
    df_ocorrencias_bronze
    .groupBy("tipo_ocorrencia")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

prioridades_validas = ["Baixa", "Média", "Alta", "Crítica"]

# Conversões necessárias
df_ocorrencias_preparado = (
    df_ocorrencias_bronze
    .withColumn(
        "_data_ocorrencia_original",
        F.col("data_ocorrencia")
    )
    .withColumn(
        "_tempo_resolucao_original",
        F.col("tempo_resolucao_minutos")
    )
    .withColumn(
        "data_ocorrencia",
        F.expr(
            "try_to_timestamp(trim(data_ocorrencia), 'yyyy-MM-dd HH:mm:ss')"
        )
    )
    .withColumn(
        "tempo_resolucao_minutos",
        F.col("tempo_resolucao_minutos")
        .cast("double")
        .cast("int")
    )
    .withColumn(
        "_mes_arquivo",
        F.regexp_extract(
            "_source_file",
            r"ocorrencias_(?:late_)?(\d{4}-\d{2})\.csv",
            1
        )
    )
    .withColumn(
        "_mes_evento",
        F.date_format("data_ocorrencia", "yyyy-MM")
    )
)

# Registro sem chave fica separado
df_ocorrencias_sem_chave = (
    df_ocorrencias_preparado
    .filter(F.col("id_ocorrencia").isNull())
)

# Critérios para escolher a melhor versão de cada ID
janela_ocorrencias = (
    Window
    .partitionBy("id_ocorrencia")
    .orderBy(
        F.col("data_ocorrencia").isNotNull().desc(),
        F.col("prioridade").isin(prioridades_validas).desc(),
        F.coalesce(
            F.col("_mes_evento") == F.col("_mes_arquivo"),
            F.lit(False)
        ).desc(),
        F.col("_ingestion_timestamp").desc(),
        F.col("_source_file").asc()
    )
)

df_ocorrencias_rankeado = (
    df_ocorrencias_preparado
    .filter(F.col("id_ocorrencia").isNotNull())
    .withColumn(
        "_row_number",
        F.row_number().over(janela_ocorrencias)
    )
)

display(
    df_ocorrencias_rankeado
    .filter(
        F.col("id_ocorrencia").isin(
            "OCR000004",
            "OCR000013",
            "OCR000020",
            "OCR000024",
            "OCR000031"
        )
    )
    .select(
        "id_ocorrencia",
        "_data_ocorrencia_original",
        "data_ocorrencia",
        "prioridade",
        "tempo_resolucao_minutos",
        "_mes_evento",
        "_mes_arquivo",
        "_source_file",
        "_row_number"
    )
    .orderBy("id_ocorrencia", "_row_number")
)

In [0]:
from pyspark.sql import functions as F

df_ocorrencias_selecionado = (
    df_ocorrencias_rankeado
    .filter(F.col("_row_number") == 1)
)

df_ocorrencias_rejeitados = (
    df_ocorrencias_rankeado
    .filter(F.col("_row_number") > 1)
    .withColumn(
        "_rejection_reason",
        F.when(
            F.col("data_ocorrencia").isNull(),
            F.lit("DATA_OCORRENCIA_INVALIDA")
        )
        .when(
            ~F.col("prioridade").isin(prioridades_validas),
            F.lit("PRIORIDADE_INVALIDA")
        )
        .otherwise(
            F.lit("DUPLICIDADE_TECNICA_ENTRE_ARQUIVOS")
        )
    )
    .unionByName(
        df_ocorrencias_sem_chave.withColumn(
            "_rejection_reason",
            F.lit("REGISTRO_DUPLICADO_SEM_CHAVE")
        ),
        allowMissingColumns=True
    )
)

print("Registros selecionados:", df_ocorrencias_selecionado.count())
print("Registros rejeitados:", df_ocorrencias_rejeitados.count())

display(
    df_ocorrencias_rejeitados
    .groupBy("_rejection_reason")
    .count()
    .orderBy("_rejection_reason")
)

In [0]:
from pyspark.sql import functions as F

SILVER_BATCH_ID = "ocorrencias_silver_2026_06_21_batch_001"

processed_timestamp = (
    spark.sql("SELECT current_timestamp() AS ts")
    .first()["ts"]
)

df_ocorrencias_silver = (
    df_ocorrencias_selecionado
    .select(
        "id_ocorrencia",
        "id_condominio",
        "id_operador",
        "data_ocorrencia",
        "tipo_ocorrencia",
        "prioridade",
        "status_ocorrencia",
        "tempo_resolucao_minutos",
        "descricao_resumida",
        "_source_file",
        "_ingestion_timestamp",
        "_ingestion_date",
        "_batch_id"
    )
    .withColumn(
        "_processed_timestamp",
        F.lit(processed_timestamp).cast("timestamp")
    )
    .withColumn(
        "_silver_batch_id",
        F.lit(SILVER_BATCH_ID)
    )
)

display(
    df_ocorrencias_silver.agg(
        F.count("*").alias("total_registros"),
        F.countDistinct("id_ocorrencia").alias("ids_distintos"),

        F.sum(
            F.when(F.col("id_ocorrencia").isNull(), 1).otherwise(0)
        ).alias("ids_ausentes"),

        F.sum(
            F.when(F.col("data_ocorrencia").isNull(), 1).otherwise(0)
        ).alias("datas_invalidas"),

        F.sum(
            F.when(
                ~F.col("prioridade").isin(prioridades_validas),
                1
            ).otherwise(0)
        ).alias("prioridades_invalidas"),

        F.sum(
            F.when(
                (F.col("status_ocorrencia") == "Resolvida")
                & F.col("tempo_resolucao_minutos").isNull(),
                1
            ).otherwise(0)
        ).alias("resolvidas_sem_tempo")
    )
)

df_ocorrencias_silver.printSchema()

In [0]:
SILVER_TABLE = "coretec_dev.silver.ocorrencias"
REJECTED_TABLE = "coretec_dev.ops.rejected_ocorrencias"

df_ocorrencias_rejeitados_final = (
    df_ocorrencias_rejeitados
    .select(
        "id_ocorrencia",
        "id_condominio",
        "id_operador",
        F.col("_data_ocorrencia_original").alias("data_ocorrencia_original"),
        "tipo_ocorrencia",
        "prioridade",
        "status_ocorrencia",
        F.col("_tempo_resolucao_original").alias("tempo_resolucao_original"),
        "descricao_resumida",
        "_rejection_reason",
        "_source_file",
        "_ingestion_timestamp",
        "_ingestion_date",
        "_batch_id"
    )
    .withColumn(
        "_processed_timestamp",
        F.lit(processed_timestamp).cast("timestamp")
    )
    .withColumn(
        "_silver_batch_id",
        F.lit(SILVER_BATCH_ID)
    )
)

(
    df_ocorrencias_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

(
    df_ocorrencias_rejeitados_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(REJECTED_TABLE)
)

print("Silver gravada:", spark.table(SILVER_TABLE).count())
print("Rejeitados gravados:", spark.table(REJECTED_TABLE).count())

In [0]:
SILVER_TABLE = "coretec_dev.silver.ocorrencias"
REJECTED_TABLE = "coretec_dev.ops.rejected_ocorrencias"

df_ocorrencias_persistido = spark.table(SILVER_TABLE)
df_rejeitados_persistido = spark.table(REJECTED_TABLE)

print("Tabela Silver existe:", spark.catalog.tableExists(SILVER_TABLE))
print("Registros Silver:", df_ocorrencias_persistido.count())

print("Tabela de rejeitados existe:", spark.catalog.tableExists(REJECTED_TABLE))
print("Registros rejeitados:", df_rejeitados_persistido.count())

print("\nSchema persistido:")
df_ocorrencias_persistido.printSchema()

print("\nFormato e histórico da Silver:")

display(
    spark.sql(f"DESCRIBE DETAIL {SILVER_TABLE}")
    .select("format", "numFiles", "sizeInBytes")
)

display(
    spark.sql(f"DESCRIBE HISTORY {SILVER_TABLE}")
    .select("version", "timestamp", "operation", "operationMetrics")
)

In [0]:
CONDOMINIOS_TABLE = "coretec_dev.bronze.condominios"

df_condominios_bronze = spark.table(CONDOMINIOS_TABLE)

print(f"Tabela: {CONDOMINIOS_TABLE}")
print(f"Quantidade de registros: {df_condominios_bronze.count()}")
print(f"Quantidade de colunas: {len(df_condominios_bronze.columns)}")
print(f"Colunas: {df_condominios_bronze.columns}")

print("\nSchema:")
df_condominios_bronze.printSchema()

display(df_condominios_bronze.limit(10))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

colunas_negocio = [
    "id_condominio",
    "nome_condominio",
    "bairro",
    "cidade",
    "uf",
    "tipo_condominio",
    "qtd_unidades",
    "plano_contratado",
    "status_contrato",
    "data_inicio_contrato",
    "valor_mensalidade_base"
]

colunas_string = [
    campo.name
    for campo in df_condominios_bronze.schema.fields
    if isinstance(campo.dataType, StringType)
    and campo.name in colunas_negocio
]

expressoes = []

for coluna in colunas_negocio:
    expressoes.append(
        F.sum(
            F.when(F.col(coluna).isNull(), 1).otherwise(0)
        ).alias(f"{coluna}_nulos")
    )

    if coluna in colunas_string:
        expressoes.append(
            F.sum(
                F.when(
                    F.col(coluna).isNotNull()
                    & (F.trim(F.col(coluna)) == ""),
                    1
                ).otherwise(0)
            ).alias(f"{coluna}_vazios")
        )

resultado_completude = df_condominios_bronze.agg(*expressoes)

display(resultado_completude)

In [0]:
display(
    df_condominios_bronze.agg(
        F.count("*").alias("total_registros"),
        F.count("id_condominio").alias("ids_nao_nulos"),
        F.countDistinct("id_condominio").alias("ids_distintos")
    )
)

df_condominios_duplicados = (
    df_condominios_bronze
    .groupBy("id_condominio")
    .count()
    .filter(F.col("count") > 1)
)

display(df_condominios_duplicados)

In [0]:
df_validade_condominios = (
    df_condominios_bronze
    .withColumn(
        "_data_inicio_teste",
        F.expr(
            "try_cast(trim(data_inicio_contrato) AS DATE)"
        )
    )
)

display(
    df_validade_condominios.agg(
        F.sum(
            F.when(
                F.col("_data_inicio_teste").isNull(),
                1
            ).otherwise(0)
        ).alias("datas_invalidas"),

        F.sum(
            F.when(
                F.col("qtd_unidades") <= 0,
                1
            ).otherwise(0)
        ).alias("quantidades_invalidas"),

        F.sum(
            F.when(
                F.col("valor_mensalidade_base") <= 0,
                1
            ).otherwise(0)
        ).alias("mensalidades_invalidas"),

        F.min("qtd_unidades").alias("menor_qtd_unidades"),
        F.max("qtd_unidades").alias("maior_qtd_unidades"),
        F.min("valor_mensalidade_base").alias("menor_mensalidade"),
        F.max("valor_mensalidade_base").alias("maior_mensalidade")
    )
)

display(
    df_validade_condominios
    .filter(
        F.col("_data_inicio_teste").isNull()
        | (F.col("qtd_unidades") <= 0)
        | (F.col("valor_mensalidade_base") <= 0)
    )
)

In [0]:
df_consistencia_condominios = (
    df_condominios_bronze
    .selectExpr(
        """
        stack(
            5,
            'cidade', cidade,
            'uf', uf,
            'tipo_condominio', tipo_condominio,
            'plano_contratado', plano_contratado,
            'status_contrato', status_contrato
        ) AS (coluna, valor)
        """
    )
    .groupBy("coluna", "valor")
    .count()
    .orderBy("coluna", F.desc("count"), "valor")
)

display(df_consistencia_condominios)

In [0]:
display(
    df_condominios_bronze
    .select(
        F.regexp_replace(
            F.trim("id_condominio"),
            r"\d",
            "9"
        ).alias("padrao_id_condominio"),

        F.length(F.trim("uf")).alias("tamanho_uf"),

        F.regexp_replace(
            F.trim("data_inicio_contrato"),
            r"\d",
            "9"
        ).alias("padrao_data_inicio")
    )
    .groupBy(
        "padrao_id_condominio",
        "tamanho_uf",
        "padrao_data_inicio"
    )
    .count()
)

In [0]:
from pyspark.sql import functions as F

SILVER_BATCH_ID = "condominios_silver_2026_06_21_batch_001"

processed_timestamp = (
    spark.sql("SELECT current_timestamp() AS ts")
    .first()["ts"]
)

df_condominios_silver = (
    df_condominios_bronze
    .select(
        "id_condominio",
        "nome_condominio",
        "bairro",
        "cidade",
        "uf",
        "tipo_condominio",

        F.col("qtd_unidades")
        .cast("int")
        .alias("qtd_unidades"),

        "plano_contratado",
        "status_contrato",

        F.to_date("data_inicio_contrato")
        .alias("data_inicio_contrato"),

        F.col("valor_mensalidade_base")
        .cast("decimal(12,2)")
        .alias("valor_mensalidade_base"),

        "_source_file",
        "_ingestion_timestamp",
        "_ingestion_date",
        "_batch_id"
    )
    .withColumn(
        "_processed_timestamp",
        F.lit(processed_timestamp).cast("timestamp")
    )
    .withColumn(
        "_silver_batch_id",
        F.lit(SILVER_BATCH_ID)
    )
)

df_condominios_silver.printSchema()

display(df_condominios_silver.limit(10))

In [0]:
display(
    df_condominios_silver.agg(
        F.count("*").alias("total_registros"),
        F.countDistinct("id_condominio").alias("ids_distintos"),

        F.sum(
            F.when(F.col("id_condominio").isNull(), 1).otherwise(0)
        ).alias("ids_ausentes"),

        F.sum(
            F.when(F.col("data_inicio_contrato").isNull(), 1).otherwise(0)
        ).alias("datas_nulas"),

        F.sum(
            F.when(F.col("qtd_unidades") <= 0, 1).otherwise(0)
        ).alias("quantidades_invalidas"),

        F.sum(
            F.when(
                F.col("valor_mensalidade_base") <= 0,
                1
            ).otherwise(0)
        ).alias("mensalidades_invalidas")
    )
)

In [0]:
SILVER_TABLE = "coretec_dev.silver.condominios"

(
    df_condominios_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"Tabela gravada: {SILVER_TABLE}")
print(f"Quantidade gravada: {spark.table(SILVER_TABLE).count()}")

In [0]:
SILVER_TABLE = "coretec_dev.silver.condominios"

df_condominios_persistido = spark.table(SILVER_TABLE)

print("Tabela existe:", spark.catalog.tableExists(SILVER_TABLE))
print("Quantidade persistida:", df_condominios_persistido.count())

print("\nSchema persistido:")
df_condominios_persistido.printSchema()

print("\nFormato e histórico Delta:")

display(
    spark.sql(f"DESCRIBE DETAIL {SILVER_TABLE}")
    .select("format", "numFiles", "sizeInBytes")
)

display(
    spark.sql(f"DESCRIBE HISTORY {SILVER_TABLE}")
    .select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics"
    )
)

display(
    df_condominios_persistido
    .orderBy("id_condominio")
    .limit(10)
)